In [1]:
import os
# Change to MultiBench directory
os.chdir('MultiBench')

In [2]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

h:\Programs\Anaconda\envs\multimodal\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load video modality
with open('data/exported_modalities/mosi_vision.pkl', 'rb') as f:
    video_data = pickle.load(f)

print("✓ Video modality loaded")
print(f"  Train: {video_data['train']['data'].shape}")
print(f"  Valid: {video_data['valid']['data'].shape}")
print(f"  Test:  {video_data['test']['data'].shape}")
print(f"  Features: 35 (Facet visual features)")

✓ Video modality loaded
  Train: (1283, 50, 35)
  Valid: (214, 50, 35)
  Test:  (686, 50, 35)
  Features: 35 (Facet visual features)


In [4]:
# Convert 7-class labels to 5-class labels
# Original: -3, -2, -1, 0, +1, +2, +3
# TFN 5-class mapping: [-3,-2] → 0, [-1] → 1, [0] → 2, [+1] → 3, [+2,+3] → 4

def convert_to_5class(labels):
    """
    Convert continuous sentiment scores to 5 classes:
    Class 0: Highly Negative (sentiment ≤ -1.5)
    Class 1: Negative (-1.5 < sentiment ≤ -0.5)
    Class 2: Neutral (-0.5 < sentiment ≤ 0.5)
    Class 3: Positive (0.5 < sentiment ≤ 1.5)
    Class 4: Highly Positive (sentiment > 1.5)
    """
    labels = labels.flatten()
    converted = np.zeros_like(labels, dtype=np.int64)
    
    converted[labels <= -1.5] = 0  # Highly Negative
    converted[(labels > -1.5) & (labels <= -0.5)] = 1  # Negative
    converted[(labels > -0.5) & (labels <= 0.5)] = 2  # Neutral
    converted[(labels > 0.5) & (labels <= 1.5)] = 3  # Positive
    converted[labels > 1.5] = 4  # Highly Positive
    
    return converted

# Convert labels
train_labels_5class = convert_to_5class(video_data['train']['labels'])
valid_labels_5class = convert_to_5class(video_data['valid']['labels'])
test_labels_5class = convert_to_5class(video_data['test']['labels'])

# Display class distribution
class_names = ['Highly Negative', 'Negative', 'Neutral', 'Positive', 'Highly Positive']
print("\n5-Class Label Distribution:")
print("=" * 80)

for split_name, labels in [('TRAIN', train_labels_5class), 
                           ('VALID', valid_labels_5class), 
                           ('TEST', test_labels_5class)]:
    print(f"\n{split_name}:")
    for class_idx, class_name in enumerate(class_names):
        count = int(np.sum(labels == class_idx))
        percentage = (count / len(labels)) * 100
        print(f"  Class {class_idx} ({class_name:<18}): {count:>4} samples ({percentage:>5.1f}%)")

print("\n✓ Labels converted to 5 classes")


5-Class Label Distribution:

TRAIN:
  Class 0 (Highly Negative   ):  223 samples ( 17.4%)
  Class 1 (Negative          ):  241 samples ( 18.8%)
  Class 2 (Neutral           ):  233 samples ( 18.2%)
  Class 3 (Positive          ):  229 samples ( 17.8%)
  Class 4 (Highly Positive   ):  357 samples ( 27.8%)

VALID:
  Class 0 (Highly Negative   ):   33 samples ( 15.4%)
  Class 1 (Negative          ):   26 samples ( 12.1%)
  Class 2 (Neutral           ):   48 samples ( 22.4%)
  Class 3 (Positive          ):   39 samples ( 18.2%)
  Class 4 (Highly Positive   ):   68 samples ( 31.8%)

TEST:
  Class 0 (Highly Negative   ):  202 samples ( 29.4%)
  Class 1 (Negative          ):  148 samples ( 21.6%)
  Class 2 (Neutral           ):  103 samples ( 15.0%)
  Class 3 (Positive          ):  114 samples ( 16.6%)
  Class 4 (Highly Positive   ):  119 samples ( 17.3%)

✓ Labels converted to 5 classes


In [5]:
# Set random seeds for reproducibility
torch.manual_seed(123)  # Changed seed for different initialization
np.random.seed(123)
if torch.cuda.is_available():
    torch.cuda.manual_seed(123)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Create PyTorch datasets
train_dataset = TensorDataset(
    torch.FloatTensor(video_data['train']['data']),
    torch.LongTensor(train_labels_5class)
)
valid_dataset = TensorDataset(
    torch.FloatTensor(video_data['valid']['data']),
    torch.LongTensor(valid_labels_5class)
)
test_dataset = TensorDataset(
    torch.FloatTensor(video_data['test']['data']),
    torch.LongTensor(test_labels_5class)
)

# Create DataLoaders with optimal batch size
batch_size = 16  # Further reduced for better generalization
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("✓ DataLoaders created")
print(f"  Batch size: {batch_size} (optimized for 5-class task)")
print(f"  Train batches: {len(train_loader)}")
print(f"  Valid batches: {len(valid_loader)}")
print(f"  Test batches: {len(test_loader)}")
print(f"  Random seed: 123 (different initialization)")

✓ DataLoaders created
  Batch size: 16 (optimized for 5-class task)
  Train batches: 80
  Valid batches: 14
  Test batches: 43
  Random seed: 123 (different initialization)


In [6]:
# Video Unimodal Model for 5-class Classification (TFN Architecture - Enhanced)
class VideoUnimodal5Class(nn.Module):
    def __init__(self, input_dim=35, lstm_hidden=128, embed_dim=256, num_classes=5, dropout=0.25):
        super(VideoUnimodal5Class, self).__init__()
        
        # 1. GRU instead of LSTM (often works better for smaller datasets)
        self.gru = nn.GRU(input_dim, lstm_hidden, batch_first=True, 
                         bidirectional=True, num_layers=2, dropout=0.15)
        self.gru_ln = nn.LayerNorm(lstm_hidden * 2)
        
        # 2. Attention mechanism for better temporal weighting
        self.attention = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        
        # 3. Visual Embedding Subnetwork
        self.embed_net = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout * 0.8)
        )
        
        # 4. Enhanced Classifier with residual connection
        self.classifier_main = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(dropout * 0.6),
        )
        
        self.classifier_out = nn.Linear(128, num_classes)
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.GRU):
                for name, param in m.named_parameters():
                    if 'weight_ih' in name:
                        nn.init.kaiming_normal_(param.data)
                    elif 'weight_hh' in name:
                        nn.init.orthogonal_(param.data)
                    elif 'bias' in name:
                        nn.init.constant_(param.data, 0)
    
    def forward(self, x):
        # GRU encoding
        gru_out, h_n = self.gru(x)  # gru_out: (batch, seq, hidden*2)
        
        # Attention mechanism
        attention_weights = torch.softmax(self.attention(gru_out), dim=1)  # (batch, seq, 1)
        attended = torch.sum(attention_weights * gru_out, dim=1)  # (batch, hidden*2)
        attended = self.gru_ln(attended)
        
        # Visual embedding
        z_v = self.embed_net(attended)
        
        # Classification
        features = self.classifier_main(z_v)
        output = self.classifier_out(features)
        
        return output

# Initialize model with enhanced architecture
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = VideoUnimodal5Class(input_dim=35, lstm_hidden=128, embed_dim=256, num_classes=5, dropout=0.25)


# Calculate class weights more aggressively
class_counts = np.bincount(train_labels_5class)
class_weights = 1.0 / (class_counts ** 0.5)  # Square root to moderate the effect
class_weights = class_weights / class_weights.sum() * len(class_weights)
class_weights = torch.FloatTensor(class_weights).to(device)

# Loss with label smoothing and class weights
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.005, betas=(0.9, 0.999))

# Cosine annealing scheduler for better convergence
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

print("✓ Enhanced Model initialized for 5-class classification")
print(f"  Device: {device}")
print(f"  Input: 35 visual features")
print(f"  Architecture: Bidirectional GRU (2 layers) + Attention")
print(f"  LSTM hidden: 128 → 256 (bidirectional)")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Loss: CrossEntropyLoss + Label Smoothing (0.1)")
print(f"  Class weights: {class_weights.cpu().numpy()}")
print(f"  Optimizer: AdamW (lr=1e-3, wd=0.005)")
print(f"  Scheduler: CosineAnnealingWarmRestarts")

✓ Enhanced Model initialized for 5-class classification
  Device: cuda
  Input: 35 visual features
  Architecture: Bidirectional GRU (2 layers) + Attention
  LSTM hidden: 128 → 256 (bidirectional)
  Parameters: 672,902
  Loss: CrossEntropyLoss + Label Smoothing (0.1)
  Class weights: [1.0599066  1.019557   1.0369123  1.0459292  0.83769494]
  Optimizer: AdamW (lr=1e-3, wd=0.005)
  Scheduler: CosineAnnealingWarmRestarts


In [7]:
# Model Summary using torchinfo (better for LSTM models)
try:
    from torchinfo import summary
    
    seq_len = video_data['train']['data'].shape[1]  # 50
    input_dim = video_data['train']['data'].shape[2]  # 300
    batch_size_summary = 2  # for summary display
    
    print("=" * 70)
    print("MODEL SUMMARY")
    print("=" * 70)
    
    summary(model, 
            input_size=(batch_size_summary, seq_len, input_dim),
            device=str(device),
            col_names=["input_size", "output_size", "num_params", "trainable"],
            row_settings=["var_names"],
            verbose=1)
    
except ImportError:
    print("⚠ torchinfo not installed. Installing...")
    print("Run: pip install torchinfo")
    print("\nAlternatively, here's a manual parameter count:")
    print("=" * 70)
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"\nTotal Parameters: {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,}")
    print(f"Non-trainable Parameters: {total_params - trainable_params:,}")
    
    print("\nLayer-wise Parameter Count:")
    print("-" * 70)
    for name, module in model.named_children():
        params = sum(p.numel() for p in module.parameters())
        print(f"{name:<20} {params:>15,} parameters")
    print("=" * 70)

MODEL SUMMARY
Layer (type (var_name))                       Input Shape               Output Shape              Param #                   Trainable
VideoUnimodal5Class (VideoUnimodal5Class)     [2, 50, 35]               [2, 5]                    --                        True
├─GRU (gru)                                   [2, 50, 35]               [2, 50, 256]              423,168                   True
├─Sequential (attention)                      [2, 50, 256]              [2, 50, 1]                --                        True
│    └─Linear (0)                             [2, 50, 256]              [2, 50, 64]               16,448                    True
│    └─Tanh (1)                               [2, 50, 64]               [2, 50, 64]               --                        --
│    └─Linear (2)                             [2, 50, 64]               [2, 50, 1]                65                        True
├─LayerNorm (gru_ln)                          [2, 256]                  [2, 256]

In [32]:
# Training and evaluation functions with mixup augmentation
def mixup_data(x, y, alpha=0.2):
    """Apply mixup augmentation."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)
    
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Calculate mixup loss."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def train_epoch(model, loader, criterion, optimizer, device, use_mixup=True):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for batch_data, batch_labels in loader:
        batch_data = batch_data.to(device)
        batch_labels = batch_labels.to(device)
        
        # Apply mixup augmentation
        if use_mixup and np.random.random() > 0.5:
            batch_data, targets_a, targets_b, lam = mixup_data(batch_data, batch_labels, alpha=0.2)
            
            optimizer.zero_grad()
            outputs = model(batch_data)
            loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
        else:
            optimizer.zero_grad()
            outputs = model(batch_data)
            loss = criterion(outputs, batch_labels)
        
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        
        optimizer.step()
        
        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_labels.cpu().numpy())
    
    avg_loss = total_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    
    return avg_loss, accuracy

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch_data, batch_labels in loader:
            batch_data = batch_data.to(device)
            batch_labels = batch_labels.to(device)
            
            outputs = model(batch_data)
            loss = criterion(outputs, batch_labels)
            
            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch_labels.cpu().numpy())
    
    avg_loss = total_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    
    return avg_loss, accuracy, all_preds, all_labels

print("✓ Training functions defined with mixup augmentation and gradient clipping")

✓ Training functions defined with mixup augmentation and gradient clipping


In [33]:
# Training loop with enhanced configuration
num_epochs = 200  # More epochs with better regularization
patience = 25
best_val_acc = 0
patience_counter = 0

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [],
    'lr': []
}

print("=" * 70)
print("TRAINING ENHANCED VIDEO UNIMODAL MODEL (5-CLASS CLASSIFICATION)")
print("=" * 70)
print(f"Epochs: {num_epochs} | Batch Size: {batch_size} | Initial LR: 1e-3")
print(f"Architecture: Bidirectional GRU (2 layers) + Attention Mechanism")
print(f"Augmentation: Mixup (alpha=0.2, prob=0.5)")
print(f"Regularization: Dropout=0.25, Weight Decay=0.005, Label Smoothing=0.1")
print(f"Scheduler: CosineAnnealingWarmRestarts | Grad Clip=0.5")
print(f"Target: Paper Accuracy = 30.4%")
print("=" * 70)

for epoch in range(num_epochs):
    # Train with mixup
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, use_mixup=True)
    
    # Validate
    val_loss, val_acc, _, _ = evaluate(model, valid_loader, criterion, device)
    
    # Update scheduler
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)
    
    # Print progress
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:3d}/{num_epochs}] | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} ({train_acc*100:.1f}%) | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} ({val_acc*100:.1f}%) | "
              f"LR: {current_lr:.6f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc
        }, 'best_video_5class.pt')
        patience_counter = 0
        print(f"  ✓ New best model saved! (Val Acc: {val_acc:.4f} = {val_acc*100:.1f}%)")
    else:
        patience_counter += 1
    
    # Early stopping
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
        break
    
    if current_lr < 1e-6 and epoch > 50:
        print(f"\nStopping: Learning rate too small ({current_lr:.2e})")
        break

print("=" * 70)
print(f"✓ Training completed!")
print(f"  Best Validation Accuracy: {best_val_acc:.4f} ({best_val_acc*100:.1f}%)")
print(f"  Total Epochs: {epoch + 1}")
print("=" * 70)

TRAINING ENHANCED VIDEO UNIMODAL MODEL (5-CLASS CLASSIFICATION)
Epochs: 200 | Batch Size: 16 | Initial LR: 1e-3
Architecture: Bidirectional GRU (2 layers) + Attention Mechanism
Augmentation: Mixup (alpha=0.2, prob=0.5)
Regularization: Dropout=0.25, Weight Decay=0.005, Label Smoothing=0.1
Scheduler: CosineAnnealingWarmRestarts | Grad Clip=0.5
Target: Paper Accuracy = 30.4%
Epoch [  1/200] | Train Loss: 3.2975 Acc: 0.2234 (22.3%) | Val Loss: 1.8491 Acc: 0.2290 (22.9%) | LR: 0.000976
  ✓ New best model saved! (Val Acc: 0.2290 = 22.9%)
Epoch [  1/200] | Train Loss: 3.2975 Acc: 0.2234 (22.3%) | Val Loss: 1.8491 Acc: 0.2290 (22.9%) | LR: 0.000976
  ✓ New best model saved! (Val Acc: 0.2290 = 22.9%)
  ✓ New best model saved! (Val Acc: 0.2897 = 29.0%)
  ✓ New best model saved! (Val Acc: 0.2897 = 29.0%)
Epoch [  5/200] | Train Loss: 1.6559 Acc: 0.2797 (28.0%) | Val Loss: 1.5947 Acc: 0.3037 (30.4%) | LR: 0.000501
  ✓ New best model saved! (Val Acc: 0.3037 = 30.4%)
Epoch [  5/200] | Train Loss: 1.

In [34]:
# Load best model and evaluate on test set
checkpoint = torch.load('best_video_5class.pt')
model.load_state_dict(checkpoint['model_state_dict'])

test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion, device)

# TFN Paper benchmark for 5-class
paper_acc = 0.304  # 30.4%

print("\n" + "=" * 70)
print("TEST SET RESULTS (Video Unimodal - 5-Class Classification)")
print("=" * 70)
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print("=" * 70)

print("\nComparison with TFN Paper (Zadeh et al., 2017):")
print("-" * 70)
print(f"{'Metric':<20} {'Our Model':<20} {'Paper':<20} {'Difference':<15}")
print("-" * 70)
print(f"{'Accuracy':<20} {test_acc:.4f} ({test_acc*100:.1f}%){'':<6} {paper_acc:.4f} ({paper_acc*100:.1f}%){'':<6} {(test_acc-paper_acc)*100:+.1f}%")
print("-" * 70)

if abs(test_acc - paper_acc) < 0.03:
    print("✓ Results are within expected range of paper benchmark!")
elif test_acc > paper_acc:
    print("✓ Results exceed paper benchmark!")
else:
    print("⚠ Results differ from paper. Consider adjusting hyperparameters.")

print("\n✓ 5-Class video evaluation completed!")


TEST SET RESULTS (Video Unimodal - 5-Class Classification)
Test Accuracy: 0.2551 (25.51%)

Comparison with TFN Paper (Zadeh et al., 2017):
----------------------------------------------------------------------
Metric               Our Model            Paper                Difference     
----------------------------------------------------------------------
Accuracy             0.2551 (25.5%)       0.3040 (30.4%)       -4.9%
----------------------------------------------------------------------
⚠ Results differ from paper. Consider adjusting hyperparameters.

✓ 5-Class video evaluation completed!
